# SQL Básico com PySpark

## Antes de começar: tipos de dados e *schema*

Quando criamos uma tabela (seja em um banco relacional tradicional ou no Spark), precisamos dizer, para cada coluna, **que tipo de dado** ela vai guardar. Isso é importante porque o tipo define:

- quanto espaço em memória/disco a coluna ocupa;
- quais operações fazem sentido nela (não dá pra fazer `AVG()` de uma coluna de texto, por exemplo);
- como os valores são comparados e ordenados.

O conjunto de colunas de uma tabela, com seus respectivos nomes e tipos, é chamado de **schema** (esquema). É basicamente a "planta baixa" da tabela. No Spark, podemos sempre visualizar o schema de um DataFrame com `df.printSchema()`.

**Principais tipos de dados que vamos usar (e que já vêm importados na célula abaixo):**

| Tipo Spark | Equivalente em SQL | Uso |
|---|---|---|
| `StringType` | `VARCHAR` / `TEXT` | Texto (nomes, cidades, categorias) |
| `IntegerType` / `LongType` | `INT` / `BIGINT` | Números inteiros |
| `FloatType` / `DoubleType` | `FLOAT` / `DOUBLE` | Números decimais (atenção: podem ter pequenos erros de arredondamento) |
| `DecimalType` | `DECIMAL(p,s)` | Números decimais **exatos** — ideal para dinheiro |
| `BooleanType` | `BOOLEAN` | Verdadeiro/falso |
| `DateType` / `TimestampType` | `DATE` / `TIMESTAMP` | Datas e datas com hora |
| `ArrayType` / `MapType` | — | Coleções (listas e dicionários dentro de uma célula) |

Na aula de hoje vamos usar principalmente `INT`, `VARCHAR` e `FLOAT`, mas é bom já conhecer os outros nomes, porque vão aparecer quando vocês lerem documentação ou mensagens de erro do Spark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, BinaryType, ArrayType, MapType, DecimalType
)
import datetime
import decimal
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

spark = SparkSession.builder.getOrCreate()

KeyboardInterrupt: 

**CRIAR TABELA**

In [ ]:
spark.sql("""
    CREATE TABLE teste (
        id INT,
        nome VARCHAR(30),
        idade INT
    )
""")

**ALTERAÇÕES NA TABELA**


*   Renomear
*   Alterar tipo de dado
*   Adicionar nova coluna
*   Deletar

In [ ]:
#Renomear coluna
df=spark.table("teste")
df=df.withColumnRenamed("idade","salário")

In [ ]:
#Alterar tipo de dado
df = df.withColumn("salário", col("salário").cast("float"))

In [ ]:
df.printSchema() #pra visu

In [ ]:
#Adicionar nova coluna
df = df.withColumn("estado civil", lit("solteiro"))

In [ ]:
spark.sql("DROP TABLE teste") #Deletar tabela (só faça quando for demitido)

**MEXER COM DADOS**

In [ ]:
spark.sql("""
    CREATE TABLE estatisticos (
        id INT,
        nome VARCHAR(30),
        profissao VARCHAR(30),
        cidade VARCHAR(30),
        salario FLOAT,
        bonus FLOAT,
        nota FLOAT

    )
""")

> **Nota sobre a coluna `nota`:** o ideal, em SQL padrão, seria declarar essa regra já na criação da coluna com uma *constraint*: `nota FLOAT CHECK (nota BETWEEN 0 AND 5)`. Isso funciona em bancos relacionais tradicionais (Postgres, MySQL etc.), mas **o catálogo padrão do Spark usado aqui no Colab não suporta `CHECK` em tabelas comuns** (ele dá erro de "feature not supported") — esse recurso só existe em formatos transacionais como o Delta Lake, que não estamos usando nesta aula. Então, na prática, aplicamos a regra "por fora": validamos os dados depois de inseridos, como na célula abaixo. É o mesmo espaço que estava reservado com a anotação "limitar superiormente a nota".

In [ ]:
#Verifica se alguma nota inserida está fora do intervalo permitido (0 a 5)
spark.sql("SELECT * FROM estatisticos WHERE nota < 0 OR nota > 5").show()

In [ ]:
spark.sql("""
    INSERT INTO estatisticos VALUES
      (2, 'Bayes', 'funileiro','sao paulo', 4000, 500, 3.9),
      (3, 'William Gosset', 'padeiro','sao carlos', 3000, 0, 4.0),
      (4, 'Mahalanobis', 'policial', 'sao paulo', 3000, 0, 3.5),
      (5, 'Forsythe', 'professor', 'sao carlos', 2500, 300, 4.1),
      (6, 'Poisson', 'pintor', 'sao carlos', 2000, 250, 2.9),
      (7, 'Cauchy', 'padeiro', 'sao paulo', 8000, 200, 4.9),
      (8, 'Viola', 'professor', 'sao carlos', 10000, 500, 5.0),
      (9, 'Kolmogorov', 'escritor', 'sao paulo', 7000, 7000, 4.8),
      (10, 'Pearson', 'padeiro', 'sao carlos', 20000, 150, 4.35),
      (11, 'Gauss', 'professor', 'sao paulo', 1750, 500, 3.75),
      (12, 'Fisher', 'pescador', 'sao carlos', 1700, 0, 2.7),
      (13, 'Chebychev', 'carteiro', 'sao paulo', 1800, 300, 4.0),
      (14, 'Markov', 'apostador', 'sao paulo', 9000, 17500, 0.0),
      (15, 'Blackwell', 'porteiro', 'sao carlos', 2000, 500, 3.0),
      (16, 'Liliefors', 'agricultor', 'sao carlos', 10000, 0, 4.5),
      (17, 'Shapiro', 'professor', 'sao carlos', 8000, 400, 3.5),
      (18, 'Kruskal', 'vidente', 'sao paulo', 30000, 0, 0.1),
      (19, 'Kendall', 'adestrador','sao carlos', 2500, 0, 4.4),
      (20, 'Friedman', 'jogador', 'sao paulo', 50000, 2500, 2.1),
      (21, 'Wilcoxon', 'caminhoneiro', 'sao carlos', 3000, 700, 3.1),
      (22, 'Neyman', 'jornalista', 'sao carlos', 4000, 250, 1.14)
""")

**CONSULTAS EM SQL**

In [ ]:
#1. Selecionando todas as colunas e dados (*)
spark.sql("SELECT * FROM estatisticos").show()

In [ ]:
#2. Escolhendo as colunas que deseja-se exibir
spark.sql("SELECT nome,salario FROM estatisticos").show()

In [ ]:
#3. Ordenando de acordo com uma variável específica (ORDER BY)
spark.sql("""SELECT nome, salario
FROM estatisticos
ORDER BY salario DESC
""").show()

In [ ]:
#4. Funções MAX e MIN
maior=spark.sql("""SELECT MAX(salario) AS maior_salario
FROM estatisticos
""")
menor=spark.sql("""SELECT MIN(salario) AS menor_salario
FROM estatisticos
""")
maior.show()
menor.show()

In [ ]:
#5. Se eu quiser ver o nome de quem possui o maior ou menor salário (LIMIT)
spark.sql("""SELECT nome
FROM estatisticos
ORDER BY salario DESC
LIMIT 1
""").show()

spark.sql("""SELECT nome, salario
FROM estatisticos
ORDER BY salario ASC
LIMIT 1
""").show()

In [ ]:
#6. Se eu quiser ver por profissão (GROUP BY)
spark.sql("""SELECT profissao,MAX(salario)
FROM estatisticos
GROUP BY profissao
""").show()

In [ ]:
#7. Salário e bônus salarial (soma de variáveis)
spark.sql("""SELECT salario+bonus AS salario_total, salario, bonus
FROM estatisticos
""").show()

botar .round()


In [ ]:
#8. Salários médios (Função average)
spark.sql("""SELECT profissao, ROUND(AVG(salario), 2) AS media_salario
FROM estatisticos
GROUP BY profissao
ORDER BY media_salario DESC
LIMIT 5
""").show()

spark.sql("""SELECT profissao, ROUND(AVG(salario+bonus), 2) AS media_salario_bonus
FROM estatisticos
GROUP BY profissao
ORDER BY media_salario_bonus DESC
LIMIT 5
""").show()

> **Sobre o `ROUND()`:** antes, `AVG(salario)` para "padeiro" retornava algo como `10333.333333333334` — um valor com muitas casas decimais, difícil de ler e sem sentido prático para dinheiro. `ROUND(expressão, 2)` arredonda o resultado para 2 casas decimais (centavos). Sempre que uma conta puder gerar dízimas (médias, divisões), vale a pena envolver em `ROUND()`.

In [ ]:
#9. Quantas profissionais há em cada área (count)
spark.sql("SELECT profissao,count(profissao) FROM estatisticos GROUP BY profissao").show()

In [ ]:
#10. Quantos recebem acima de 10000 (clausula WHERE)
spark.sql("""SELECT nome,salario
FROM estatisticos
WHERE salario>10000
""").show()

In [ ]:
#10.1 Mais usos do WHERE
spark.sql("""SELECT ROUND(AVG(salario), 2) AS media_salario, profissao
FROM estatisticos
WHERE profissao='professor'
GROUP BY profissao
""").show()